# 冬休みの課題（囚人のジレンマ）の解答例と考察

In [1]:
from pd import *
import numpy as np

## Pavlov と第10回の戦略

Pavlov は「得点がよければ同じ手，悪ければ手を変える」戦略で，第10回の TsuikiStrategy（当たったら同じスロット，はずれたら次のスロット）と同じ win-stay, lose-shift の考え方です。

## ノイズとしっぺ返し

ノイズがないと，しっぺ返しどうしはずっと協調して 3 点をとります。ノイズで一度 D が出ると，D と C が交互に入れ替わる報復の連鎖が始まり，もう一度ノイズが起きるまで続きます。

In [2]:
for nz in [0, 0.02, 0.05]:
    r = [Match(TitForTat, TitForTat, noise=nz, seed=t)[0] for t in range(200)]
    g = [Match(GenerousTFT, GenerousTFT, noise=nz, seed=t)[0] for t in range(200)]
    p = [Match(Pavlov, Pavlov, noise=nz, seed=t)[0] for t in range(200)]
    print(f'noise={nz}: TFT同士 {np.mean(r):.3f}  GenerousTFT同士 {np.mean(g):.3f}  Pavlov同士 {np.mean(p):.3f}')

noise=0: TFT同士 3.000  GenerousTFT同士 3.000  Pavlov同士 3.000
noise=0.02: TFT同士 2.549  GenerousTFT同士 2.952  Pavlov同士 2.917
noise=0.05: TFT同士 2.461  GenerousTFT同士 2.867  Pavlov同士 2.775


寛容なしっぺ返しと Pavlov は，ノイズによる報復の連鎖から抜け出せるので，ノイズがあっても高い得点を保ちます。

## 戦略の例

相手の手の履歴を記録して，相手の性質を推測する戦略です。

* 10 回以上対戦したら，相手の協調の割合と，こちらが裏切った直後に相手が協調した割合を調べる。
  * こちらが裏切っても相手がほぼ協調し続ける（報復しない）なら，裏切って搾取する（AllC など）。
  * 相手の協調が 3〜7 割程度で，こちらの協調が報われない（ランダムや交互）なら，裏切る。
  * 相手がほとんど裏切る（AllD）なら，裏切る。
* それ以外は，寛容なしっぺ返しで応じる。

In [3]:
class ExamplePD:
    def __init__(self):
        self.h = []      # (自分の手, 相手の手) の履歴
        self.my = 'C'

    def choice(self):
        n = len(self.h)
        if n >= 10:
            oc = sum(1 for m, o in self.h if o == 'C') / n    # 相手の協調の割合
            after = [self.h[i + 1][1] for i in range(n - 1) if self.h[i][0] == 'D']
            if after and sum(1 for x in after if x == 'C') / len(after) > 0.7 and oc > 0.7:
                return 'D'   # 裏切っても報復してこない
            if 0.3 < oc < 0.7 and n >= 20:
                rc = [self.h[i + 1][1] for i in range(n - 1) if self.h[i][0] == 'C']
                if rc and sum(1 for x in rc if x == 'C') / len(rc) < 0.6:
                    return 'D'   # 協調しても報われない
            if oc < 0.2:
                return 'D'   # ほとんど裏切ってくる
        return self.my

    def learn(self, my, other):
        self.h.append((my, other))
        if other == 'D' and random.uniform(0, 1) < 1 / 3:
            self.my = 'C'
        else:
            self.my = other

In [4]:
res, total = Evaluate(ExamplePD)

          AllC : 3.928
          AllD : 0.950
       RandomS : 2.462
     Alternate : 2.254
     TitForTat : 2.858
 SuspiciousTFT : 2.710
   GenerousTFT : 2.952
 TitForTwoTats : 2.963
          Grim : 1.869
        Pavlov : 2.739
          Joss : 2.486
           平均 : 2.561
         対戦点 : 10.0 / 10


しっぺ返し（2.352）と比べると，AllC と RandomS からの搾取に加え，寛容さによって TitForTat, SuspiciousTFT, Pavlov, Joss との報復の連鎖を抜け出せることが効いています。一方，Alternate には，しっぺ返しより少し負けています。

学生どうしの大会（主催者の戦略を含む総当たり）では，どうなるでしょうか。

In [5]:
S = HOUSE + [ExamplePD]
M, total = Tournament(S)
ShowTournament(S, M, total)

     ExamplePD : 2.596
   GenerousTFT : 2.430
     Alternate : 2.410
 TitForTwoTats : 2.370
        Pavlov : 2.367
     TitForTat : 2.366
          Grim : 2.344
 SuspiciousTFT : 2.312
          Joss : 2.300
       RandomS : 2.201
          AllC : 2.147
          AllD : 2.110


## 乱数を使わない寛容なしっぺ返し

相手が固定された戦略（確率的でもよい）である時，平均得点を最大にする戦略は乱数を使わないものの中にある（各時点で C を出す確率 q について，期待得点が q の1次式になるので，q = 0 か 1 のどちらかで最大になる）。

ただし，これは「一番よい戦略は乱数を使わずに作れる」ということで，乱数を使う戦略を適当に乱数なしに書き換えれば良くなる，ということではない。

寛容なしっぺ返しの「1/3 の確率で許す」を，「3回に1回許す」に変えて比べてみよう。

In [6]:
class GenerousTFT3:
    '''相手の裏切りを 3 回に 1 回許すしっぺ返し（乱数を使わない）'''
    def __init__(self):
        self.my = 'C'
        self.count = 0

    def choice(self):
        return self.my

    def learn(self, my, other):
        if other == 'D':
            self.count += 1
            self.my = 'C' if self.count % 3 == 0 else 'D'
        else:
            self.my = 'C'

for S in [GenerousTFT, GenerousTFT3]:
    res, total = Evaluate(S, show=False)
    print(f'{S.__name__:>14s} : {total:.3f}  対戦点 {Points(total)}')

   GenerousTFT : 2.394  対戦点 6.8
  GenerousTFT3 : 2.376  対戦点 6.5


この場合は，乱数を使わない方が少し悪くなった。「3回に1回」は，相手が裏切り続ける時にも規則的に許してしまうので，許すタイミングを相手に合わせて選ぶ（例えば，ノイズによる裏切りらしい時だけ許す）ことが必要になる。最適な乱数なしの戦略は，相手の履歴全体に応じて手を選ぶものである。